In [3]:
from ultralytics import YOLO

model = YOLO(r"C:\Users\ASIF AHAMMAD\Downloads\chest-xray-detection\models\best.pt")
print("Classes:", model.names)

Classes: {0: 'Aortic enlargement', 1: 'Atelectasis', 2: 'Calcification', 3: 'Cardiomegaly', 4: 'Consolidation', 5: 'ILD', 6: 'Infiltration', 7: 'Lung Opacity', 8: 'Nodule-Mass', 9: 'Other lesion', 10: 'Pleural effusion', 11: 'Pleural thickening', 12: 'Pneumothorax', 13: 'Pulmonary fibrosis'}


# Building the CNN Dataset — Cropping YOLO Boxes

In [1]:
import os
from PIL import Image
import glob

# ---- EDIT THESE THREE PATHS TO MATCH YOUR REPO ----
YOLO_DATASET_DIR = r"C:\Users\ASIF AHAMMAD\Downloads\chest-xray-detection\datasets\Chest Xray YOLOsplit"   # has train/ and valid/
CNN_OUTPUT_DIR   = r"C:\Users\ASIF AHAMMAD\Downloads\chest-xray-detection\datasets\Cnn"    # where cropped images will go
OLD_NORMAL_DIR   = r"C:\Users\ASIF AHAMMAD\Downloads\PREPARED_CHEST_XRAY_DATASET\CNN"       # old dataset, for the Normal class
# -----------------------------------------------------

CLASS_NAMES = [
    "Aortic enlargement", "Atelectasis", "Calcification", "Cardiomegaly",
    "Consolidation", "ILD", "Infiltration", "Lung Opacity", "Nodule-Mass",
    "Other lesion", "Pleural effusion", "Pleural thickening", "Pneumothorax",
    "Pulmonary fibrosis"
]

MIN_BOX_SIZE = 10   # pixels; skip boxes smaller than this (likely annotation noise)
PADDING = 0.10      # add 10% padding around each box, so the CNN sees a little context

print("Will crop", len(CLASS_NAMES), "abnormality classes, plus Normal.")

Will crop 14 abnormality classes, plus Normal.


In [4]:
def yolo_box_to_pixels(cx, cy, w, h, img_w, img_h, padding=0.0):
    """Convert a YOLO-format normalized box (center x, center y, width, height)
    into pixel coordinates (left, top, right, bottom), with optional padding."""
    box_w = w * img_w
    box_h = h * img_h
    cx_px = cx * img_w
    cy_px = cy * img_h

    box_w *= (1 + padding)
    box_h *= (1 + padding)

    left = max(0, int(cx_px - box_w / 2))
    top = max(0, int(cy_px - box_h / 2))
    right = min(img_w, int(cx_px + box_w / 2))
    bottom = min(img_h, int(cy_px + box_h / 2))

    return left, top, right, bottom


def crop_split(split_name):
    """Crop all boxes from one split (train or valid) into CNN_OUTPUT_DIR/split_name/<class>/"""
    img_dir = os.path.join(YOLO_DATASET_DIR, split_name, "images")
    lbl_dir = os.path.join(YOLO_DATASET_DIR, split_name, "labels")
    label_files = glob.glob(os.path.join(lbl_dir, "*.txt"))

    counts = {name: 0 for name in CLASS_NAMES}
    skipped = 0

    for lbl_path in label_files:
        stem = os.path.splitext(os.path.basename(lbl_path))[0]
        img_candidates = glob.glob(os.path.join(img_dir, stem + ".*"))
        if not img_candidates:
            continue
        img_path = img_candidates[0]

        with Image.open(img_path) as im:
            im = im.convert("RGB")
            img_w, img_h = im.size

            with open(lbl_path) as fh:
                lines = [l.strip() for l in fh if l.strip()]

            for i, line in enumerate(lines):
                parts = line.split()
                cls_id = int(parts[0])
                cx, cy, w, h = map(float, parts[1:5])

                left, top, right, bottom = yolo_box_to_pixels(
                    cx, cy, w, h, img_w, img_h, padding=PADDING
                )

                if (right - left) < MIN_BOX_SIZE or (bottom - top) < MIN_BOX_SIZE:
                    skipped += 1
                    continue

                crop = im.crop((left, top, right, bottom))
                class_name = CLASS_NAMES[cls_id]
                out_dir = os.path.join(CNN_OUTPUT_DIR, split_name, class_name)
                os.makedirs(out_dir, exist_ok=True)

                out_path = os.path.join(out_dir, f"{stem}_{i}.jpg")
                crop.save(out_path, quality=95)
                counts[class_name] += 1

    print(f"--- {split_name} ---")
    for name, c in counts.items():
        print(f"{name:25s} {c}")
    print(f"Skipped (too small): {skipped}")
    return counts

In [5]:
train_counts = crop_split("train")
valid_counts = crop_split("valid")

--- train ---
Aortic enlargement        12099
Atelectasis               393
Calcification             1233
Cardiomegaly              9010
Consolidation             887
ILD                       1435
Infiltration              2152
Lung Opacity              4287
Nodule-Mass               4237
Other lesion              3663
Pleural effusion          4167
Pleural thickening        8313
Pneumothorax              364
Pulmonary fibrosis        7714
Skipped (too small): 3
--- valid ---
Aortic enlargement        1558
Atelectasis               48
Calcification             158
Cardiomegaly              1159
Consolidation             93
ILD                       160
Infiltration              265
Lung Opacity              526
Nodule-Mass               425
Other lesion              418
Pleural effusion          536
Pleural thickening        1040
Pneumothorax              51
Pulmonary fibrosis        922
Skipped (too small): 0


In [6]:
import shutil

def copy_normal_class(old_split_name, new_split_name):
    """Copy the old dataset's NORMAL_BY_ANNOTATION folder into the new CNN dataset as the 'Normal' class."""
    src_dir = os.path.join(OLD_NORMAL_DIR, old_split_name, "NORMAL_BY_ANNOTATION")
    dst_dir = os.path.join(CNN_OUTPUT_DIR, new_split_name, "Normal")
    os.makedirs(dst_dir, exist_ok=True)

    if not os.path.isdir(src_dir):
        print(f"WARNING: {src_dir} not found — check the folder name/path.")
        return 0

    count = 0
    for fname in os.listdir(src_dir):
        shutil.copy(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))
        count += 1

    print(f"{new_split_name}/Normal: copied {count} images")
    return count

copy_normal_class("train", "train")
copy_normal_class("valid", "valid")

for split in ["train", "valid"]:
    print(f"=== {split} ===")
    split_dir = os.path.join(CNN_OUTPUT_DIR, split)
    for class_name in sorted(os.listdir(split_dir)):
        class_path = os.path.join(split_dir, class_name)
        if os.path.isdir(class_path):
            n = len(os.listdir(class_path))
            print(f"{class_name:25s} {n}")

train/Normal: copied 85 images
valid/Normal: copied 9 images
=== train ===
Aortic enlargement        12099
Atelectasis               393
Calcification             1233
Cardiomegaly              9010
Consolidation             887
ILD                       1435
Infiltration              2152
Lung Opacity              4287
Nodule-Mass               4237
Normal                    85
Other lesion              3663
Pleural effusion          4167
Pleural thickening        8313
Pneumothorax              364
Pulmonary fibrosis        7714
=== valid ===
Aortic enlargement        1558
Atelectasis               48
Calcification             158
Cardiomegaly              1159
Consolidation             93
ILD                       160
Infiltration              265
Lung Opacity              526
Nodule-Mass               425
Normal                    9
Other lesion              418
Pleural effusion          536
Pleural thickening        1040
Pneumothorax              51
Pulmonary fibrosis        922
